In [1]:
!pip install transformers datasets

In [2]:
import pandas as pd
import numpy as np
import random
import time
import datetime
import csv
import os
import torch
import torch.nn.functional as F

from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from datasets import load_dataset
from torch.optim import AdamW
from transformers import pipeline
from transformers import BertTokenizer
from transformers import BertForSequenceClassification, BertConfig
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, hamming_loss

In [3]:
!git clone https://github.com/adlnlp/K-MHaS.git
!ls -al K-MHaS
!ls -al K-MHaS/data

Cloning into 'K-MHaS'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 87 (delta 23), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 11.94 MiB | 10.32 MiB/s, done.
Resolving deltas: 100% (23/23), done.
total 28
drwxr-xr-x 5 root root 4096 Feb  4 12:13 .
drwxr-xr-x 1 root root 4096 Feb  4 12:13 ..
drwxr-xr-x 2 root root 4096 Feb  4 12:13 data
drwxr-xr-x 8 root root 4096 Feb  4 12:13 .git
-rw-r--r-- 1 root root 7776 Feb  4 12:13 README.md
drwxr-xr-x 2 root root 4096 Feb  4 12:13 resource
total 9288
drwxr-xr-x 2 root root    4096 Feb  4 12:13 .
drwxr-xr-x 5 root root    4096 Feb  4 12:13 ..
-rw-r--r-- 1 root root 1902352 Feb  4 12:13 kmhas_test.txt
-rw-r--r-- 1 root root 6845463 Feb  4 12:13 kmhas_train.txt
-rw-r--r-- 1 root root  748899 Feb  4 12:13 kmhas_valid.txt


In [4]:
!head -n 3 /content/K-MHaS/data/kmhas_train.txt

document	label
"자한당틀딱들.. 악플질 고만해라."	2,4
정치적으로 편향된 평론한은 분은 별로...	8


In [5]:
train = load_dataset('csv', data_files='/content/K-MHaS/data/kmhas_train.txt', delimiter='\t', split='train')
validation = load_dataset('csv', data_files='/content/K-MHaS/data/kmhas_valid.txt', delimiter='\t', split='train')
test = load_dataset('csv', data_files='/content/K-MHaS/data/kmhas_test.txt', delimiter='\t', split='train')

print(train)
print(validation)
print(test)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['document', 'label'],
    num_rows: 78977
})
Dataset({
    features: ['document', 'label'],
    num_rows: 8776
})
Dataset({
    features: ['document', 'label'],
    num_rows: 21939
})


In [6]:
train[0]

{'document': '자한당틀딱들.. 악플질 고만해라.', 'label': '2,4'}

In [7]:
train[0]['document']

'자한당틀딱들.. 악플질 고만해라.'

In [8]:
train[0]['label']

'2,4'

In [9]:
train_sentences = list(map(lambda x: '[CLS] ' + str(x) + ' [SEP]', train['document']))
validation_sentences = list(map(lambda x: '[CLS] ' + str(x) + ' [SEP]', validation['document']))
test_sentences = list(map(lambda x: '[CLS] ' + str(x) + ' [SEP]', test['document']))

In [10]:
train_sentences[:5]

['[CLS] 자한당틀딱들.. 악플질 고만해라. [SEP]',
 '[CLS] 정치적으로 편향된 평론한은 분은 별로... [SEP]',
 '[CLS] 적당히좀 쳐먹지.그랬냐??? 안그래도 문재인 때문에 나라 엉망진창인데... [SEP]',
 '[CLS] 안서는 아재들 풀발기 ㅋㄲㅋ [SEP]',
 '[CLS] 우와 ㅋ 능력자 [SEP]']

In [11]:
def parse_label(s):
    return [int(x) for x in str(s).split(',')]

In [12]:
train_y = [parse_label(s) for s in train['label']]
validation_y = [parse_label(s) for s in validation['label']]
test_y = [parse_label(s) for s in test['label']]

In [13]:
train_y

[[2, 4],
 [8],
 [2],
 [4],
 [8],
 [8],
 [8],
 [8],
 [3],
 [2, 3],
 [8],
 [8],
 [3],
 [8],
 [8],
 [8],
 [3],
 [0],
 [8],
 [8],
 [3],
 [8],
 [3],
 [8],
 [8],
 [3],
 [3],
 [3],
 [8],
 [1, 5],
 [8],
 [5],
 [0],
 [3],
 [0],
 [8],
 [1, 5],
 [0],
 [3],
 [0],
 [7],
 [8],
 [2],
 [8],
 [8],
 [8],
 [2, 7],
 [8],
 [8],
 [3],
 [3],
 [8],
 [1],
 [8],
 [1],
 [4],
 [3, 4],
 [1, 5],
 [8],
 [5],
 [8],
 [1],
 [2],
 [8],
 [8],
 [3],
 [8],
 [8],
 [8],
 [8],
 [8],
 [5],
 [2],
 [8],
 [5],
 [2],
 [8],
 [3],
 [8],
 [8],
 [8],
 [8],
 [8],
 [8],
 [1, 3],
 [0],
 [4],
 [8],
 [0, 2],
 [8],
 [3],
 [8],
 [2, 3, 4],
 [2, 3],
 [3, 5],
 [3, 5],
 [3],
 [0],
 [0, 3],
 [8],
 [8],
 [2],
 [8],
 [8],
 [8],
 [8],
 [2],
 [5],
 [8],
 [1],
 [2],
 [2, 3],
 [4],
 [3],
 [8],
 [2],
 [8],
 [0],
 [3],
 [3],
 [8],
 [8],
 [8],
 [0],
 [3],
 [0, 5],
 [8],
 [8],
 [8],
 [8],
 [8],
 [8],
 [5],
 [8],
 [8],
 [2, 4],
 [8],
 [3, 5],
 [8],
 [3],
 [8],
 [8],
 [0],
 [8],
 [0, 2],
 [8],
 [8],
 [2],
 [8],
 [8],
 [8],
 [2],
 [0, 5],
 [8],
 [2, 3],
 [0]

In [14]:
enc = MultiLabelBinarizer()
enc.fit(train_y)
num_labels = len(enc.classes_)
num_labels, enc.classes_

(9, array([0, 1, 2, 3, 4, 5, 6, 7, 8]))

In [15]:
train_labels = torch.tensor(enc.transform(train_y), dtype=torch.float32)
validation_labels = torch.tensor(enc.transform(validation_y), dtype=torch.float32)
test_labels = torch.tensor(enc.transform(test_y), dtype=torch.float32)

In [16]:
train_labels

tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [1., 0., 0.,  ..., 0., 0., 0.]])

In [17]:
test_sentences[:5]

['[CLS] 그만큼 길예르모가 잘했다고 보면되겠지 기대되네 셰이프 오브 워터 [SEP]',
 '[CLS] 1. 8넘의 문재앙 [SEP]',
 '[CLS] 문재인 정권의 내로남불은 타의 추종을 불허하네. 자한당 욕할거리도 없음. [SEP]',
 '[CLS] 짱개들 지나간 곳은 폐허된다 ㅋㅋ [SEP]',
 '[CLS] 곱창은 자갈치~~~~~ [SEP]']

In [18]:
test[:5]

{'document': ['그만큼 길예르모가 잘했다고 보면되겠지 기대되네 셰이프 오브 워터',
  '1. 8넘의 문재앙',
  '문재인 정권의 내로남불은 타의 추종을 불허하네. 자한당 욕할거리도 없음.',
  '짱개들 지나간 곳은 폐허된다 ㅋㅋ',
  '곱창은 자갈치~~~~~'],
 'label': ['8', '2,3', '2', '0', '8']}

In [19]:
test_labels[:5]

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 1., 1., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1.]])

In [20]:
tokenizer = BertTokenizer.from_pretrained('klue/bert-base')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [21]:
MAX_LEN = 128

def data_to_tensor(sentences, labels, tokenizer, max_len=MAX_LEN):
    pad_id = tokenizer.pad_token_type_id
    input_ids = []
    for sent in sentences:
        sent = "" if sent is None else str(sent)
        if not sent.startswith('[CLS]'):
            sent = '[CLS] ' + sent
        if not sent.endswith('[SEP]'):
            sent = sent + ' [SEP]'

        tokens = tokenizer.tokenize(sent)

        if len(tokens) > max_len:
            tokens = tokens[:max_len]
            if tokens[-1] != '[SEP]':
                tokens[-1] = '[SEP]'
        ids = tokenizer.convert_tokens_to_ids(tokens)
        input_ids.append(torch.tensor(ids, dtype=torch.long))

    padded_inputs = pad_sequence(input_ids, batch_first=True, padding_value=pad_id)

    if padded_inputs.size(1) < max_len:
        padded_inputs = F.pad(padded_inputs, (0, max_len - padded_inputs.size(1)), value=pad_id)
    else:
        padded_inputs = padded_inputs[:, :max_len]

    attention_masks = (padded_inputs != pad_id).float()

    if torch.is_tensor(labels):
        tensor_labels = labels.to(dtype=torch.float32)
    else:
        tensor_labels = torch.tensor(labels, dtype=torch.float32)

    return padded_inputs, tensor_labels, attention_masks

In [22]:
train_inputs, train_labels, train_mask = data_to_tensor(train_sentences, train_labels, tokenizer)
validation_inputs, validation_labels, validation_mask = data_to_tensor(validation_sentences, validation_labels, tokenizer)
test_inputs, test_labels, test_mask = data_to_tensor(test_sentences, test_labels, tokenizer)

In [23]:
train_inputs, train_labels, train_mask

(tensor([[    2,  1517,  2470,  ...,     0,     0,     0],
         [    2,  3713, 11187,  ...,     0,     0,     0],
         [    2, 11466,  2505,  ...,     0,     0,     0],
         ...,
         [    2, 31035,  2425,  ...,     0,     0,     0],
         [    2,     1, 10758,  ...,     0,     0,     0],
         [    2,  1439,  2075,  ...,     0,     0,     0]]),
 tensor([[0., 0., 1.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 1.],
         [0., 0., 1.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 1.],
         [0., 0., 0.,  ..., 0., 0., 1.],
         [1., 0., 0.,  ..., 0., 0., 0.]]),
 tensor([[1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         ...,
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.]]))

In [24]:
batch_size = 32

train_data = TensorDataset(train_inputs, train_mask, train_labels)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

validation_data = TensorDataset(validation_inputs, validation_mask, validation_labels)
validation_sampler = RandomSampler(validation_data)
validation_dataloader = DataLoader(validation_data, sampler=validation_sampler, batch_size=batch_size)

test_data = TensorDataset(test_inputs, test_mask, test_labels)
test_sampler = RandomSampler(test_data)
test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=batch_size)

len(train_labels), len(validation_labels), len(test_labels)

(78977, 8776, 21939)

In [25]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [27]:
num_labels = 9
model = BertForSequenceClassification.from_pretrained('klue/bert-base', num_labels=num_labels,
                                                      problem_type='multi_label_classification')
model.cuda()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [33]:
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
epochs = 10
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [34]:
def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [35]:
def multi_label_metrics(predictions, labels, threshold=0.5):
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= threshold)] = 1
    y_true = labels

    accuracy = accuracy_score(y_true, y_pred)
    f1_macro_average = f1_score(y_true=y_true, y_pred=y_pred, average='macro', zero_division=0)
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro', zero_division=0)
    f1_weighted_average = f1_score(y_true=y_true, y_pred=y_pred, average='weighted', zero_division=0)
    roc_auc = roc_auc_score(y_true, y_pred, average='micro')

    metrics = {
        'accuracy': accuracy,
        'f1_macro_average': f1_macro_average,
        'f1_micro_average': f1_micro_average,
        'f1_weighted_average': f1_weighted_average,
        'roc_auc': roc_auc
    }
    return metrics

In [36]:
seed_val = 2026
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed(seed_val)

In [37]:
model.zero_grad()

for epoch_i in range(0, epochs):
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
    t0 = time.time()
    total_loss = 0

    model.train()

    for step, batch in tqdm(enumerate(train_dataloader)):
        if step % 500 == 0 and not step == 0:
            elapsed = format_time(time.time() - t0)
            print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(train_dataloader), elapsed))

        batch = tuple(t.to(device) for t in batch)
        b_input_ids, b_input_mask, b_labels = batch
        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)
        loss = outputs[0]
        total_loss += loss.item()
        loss.backward()
        # L2 norm이 1.0을 넘으면 강제로 잘라냄
        # BERT 학습에서 필수 코드
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        model.zero_grad()
    avg_train_loss = total_loss / len(train_dataloader)
    print("")
    print("  Average training loss: {0:.4f}".format(avg_train_loss))
    print("  Training epoch took: {:}".format(format_time(time.time() - t0)))

======== Epoch 1 / 10 ========


500it [02:25,  3.44it/s]

  Batch   500  of  2,469.    Elapsed: 0:02:25.


1000it [04:51,  3.44it/s]

  Batch 1,000  of  2,469.    Elapsed: 0:04:51.


...

  Batch 2,000  of  2,469.    Elapsed: 0:09:43.


2469it [11:59,  3.43it/s]



  Average training loss: 0.0862
  Training epoch took: 0:11:59
======== Epoch 3 / 10 ========


8it [00:02,  3.26it/s]


KeyboardInterrupt: 

In [ ]:
t0 = time.time()
model.eval()
accum_logits, accum_label_ids = [], []

for batch in validation_dataloader:
    batch = tuple(t.to(device) for t in batch)
    b_input_ids, b_input_mask, b_labels = batch

    with torch.no_grad():
        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask)

    logits = outputs[0]
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    for b in logits:
        accum_logits.append(list(b))

    for b in label_ids:
        accum_label_ids.append(list(b))

accum_logits = np.array(accum_logits)
accum_label_ids = np.array(accum_label_ids)
results = multi_label_metrics(accum_logits, accum_label_ids)

print("Accuracy: {0:.4f}".format(results['accuracy']))
print("F1 (Macro) Score: {0:.4f}".format(results['f1_macro_average']))
print("F1 (Micro) Score: {0:.4f}".format(results['f1_micro_average']))
print("F1 (Weighted) Score: {0:.4f}".format(results['f1_weighted_average']))
print("ROC-AUC: {0:.4f}".format(results['roc_auc']))

In [ ]:
%pwd
%mkdir model

In [ ]:
path = '/content/model/'
torch.save(model.state_dict(), path + 'BERT_multilabel_model.pt')

In [ ]:
model.load_state_dict(torch.load(path + 'BERT_multilabel_model.pt'))

In [ ]:
t0 = time.time()
model.eval()
accum_logits, accum_label_ids = [], []

for step, batch in tqdm(enumerate(test_dataloader)):
    if step % 100 == 0 and not step == 0:
        elapsed = format_time(time.time() - t0)
        print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(test_dataloader), elapsed))

    batch = tuple(t.to(device) for t in batch)
    b_input_ids, b_input_mask, b_labels = batch

    with torch.no_grad():
        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask)

    logits = outputs[0]
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    for b in logits:
        accum_logits.append(list(b))

    for b in label_ids:
        accum_label_ids.append(list(b))

accum_logits = np.array(accum_logits)
accum_label_ids = np.array(accum_label_ids)
results = multi_label_metrics(accum_logits, accum_label_ids)

print("Accuracy: {0:.4f}".format(results['accuracy']))
print("F1 (Macro) Score: {0:.4f}".format(results['f1_macro_average']))
print("F1 (Micro) Score: {0:.4f}".format(results['f1_micro_average']))
print("F1 (Weighted) Score: {0:.4f}".format(results['f1_weighted_average']))
print("ROC-AUC: {0:.4f}".format(results['roc_auc']))

In [ ]:
pipe = pipeline('text-classification', model=model.cuda(), tokenizer=tokenizer,
                device=0, max_length=512, function_to_apply='sigmoid', return_all_scores=True)

In [ ]:
result = pipe('잼민이들 왜 그렇게 민폐냐?')
print(result)

In [ ]:
label_dict = {'LABEL_0' : '출신차별', 'LABEL_1' : '외모차별', 'LABEL_2' : '정치성향차별', \
              'LABEL_3': '혐오욕설', 'LABEL_4': '연령차별', 'LABEL_5': '성차별', 'LABEL_6' : '인종차별', \
              'LABEL_7': '종교차별', 'LABEL_8': '해당사항없음'}

In [ ]:
def prediction(text):
    result = pipe(text)
    return [label_dict[res['label']] for res in result[0] if res['score'] > 0.5]

In [ ]:
prediction('너희 나라로 돌아가라')